# Phase 4 Layout Detection Training: YOLO Handwritten-Region Cropper

The heuristic cropper is only a fallback and will be inaccurate on many prescriptions. This notebook trains a YOLO layout detector so the system learns where the handwritten prescription area is.

## What this notebook does

1. Uses corrected handwritten-region boxes from your annotation/cropping workflow.
2. Converts those boxes into YOLO format.
3. Trains a YOLO model for `handwritten_region` detection.
4. Validates the trained model.
5. Runs the final pipeline using the trained detector instead of heuristic cropping.

## Important class-id note

- A model trained by this notebook has one class: `handwritten_region`, so use `--target-class 0` during inference.
- CVAT labels using `header, handwritten_region, footer` usually have `handwritten_region` as class id `1`.


## Step 0: Set Project Root

Run this from the repository root, or change `REPO_DIR` to the folder containing `pipeline/` and `data/`.


In [ ]:
# Colab/Drive project setup.
# This cell mounts Google Drive, keeps the repository in Drive, and keeps data in Drive.
# Put raw prescriptions here after this cell runs:
# /content/drive/MyDrive/phase4_project/repo/data/raw

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

    if not (REPO_DIR / 'pipeline').exists():
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            raise RuntimeError(
                f'{REPO_DIR} exists but does not look like the project repo. '
                'Move/rename it or set REPO_DIR to the correct folder.'
            )
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    # Local Jupyter/VS Code fallback: use the current repository folder.
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)

# data/ is gitignored, so it is safe to keep raw images and generated files here.
# In Colab this folder is inside Google Drive and persists across sessions.
DATA_DIR = REPO_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('Repository:', REPO_DIR)
print('Data folder:', DATA_DIR)
print('Current working directory:', Path.cwd())
print('Has pipeline:', (Path.cwd() / 'pipeline').exists())
print('Raw data folder:', RAW_DIR)
print('Raw images currently:', len([p for p in RAW_DIR.glob('*') if p.is_file()]))

## Step 1: Install YOLO/Layout Dependencies

This installs the base image pipeline dependencies plus Ultralytics YOLO.


In [ ]:
!python3 -m pip install -q -r pipeline/requirements-layout.txt


## Step 2: Prepare or Correct Region Annotations

You need accurate boxes before training. You can create them in two ways:

- Recommended: use CVAT/Label Studio and export YOLO labels.
- Simpler: use the manual region cropper in `phase4_full_prescription_annotation_tools.ipynb`, then correct bad crops and save `region_manifest.csv`.

The next cell assumes these files exist:

- `data/processed/page_manifest.csv`
- `data/processed/region_manifest.csv`

If you used another output folder, update the paths.


In [ ]:
PAGE_MANIFEST = Path('data/processed/page_manifest.csv')
REGION_MANIFEST = Path('data/processed/region_manifest.csv')
YOLO_LAYOUT_DATASET = Path('data/layout_yolo_dataset')

print('PAGE_MANIFEST exists:', PAGE_MANIFEST.exists(), PAGE_MANIFEST)
print('REGION_MANIFEST exists:', REGION_MANIFEST.exists(), REGION_MANIFEST)


## Step 3: Convert Region Boxes to YOLO Dataset

This creates `images/train`, `images/val`, `labels/train`, `labels/val`, and `data.yaml`.


In [ ]:
!python3 pipeline/scripts/prepare_yolo_layout_dataset.py \
  --page-manifest "{PAGE_MANIFEST}" \
  --region-manifest "{REGION_MANIFEST}" \
  --output-dir "{YOLO_LAYOUT_DATASET}" \
  --class-name handwritten_region \
  --val-ratio 0.2 \
  --seed 42


## Step 4: Train YOLO Handwritten-Region Detector

Use a small model (`yolov8n.pt`) for fast training. Increase epochs after your labels are clean.


In [ ]:
!python3 pipeline/scripts/train_yolo_layout.py \
  --data-yaml data/layout_yolo_dataset/data.yaml \
  --model yolov8n.pt \
  --epochs 50 \
  --imgsz 960 \
  --batch 8 \
  --project runs/layout \
  --name handwritten_region_yolo


## Step 5: Locate Best Model

The trained detector is saved as `best.pt`. Use this in the full pipeline.


In [ ]:
BEST_LAYOUT_MODEL = Path('runs/layout/handwritten_region_yolo/weights/best.pt')
print('BEST_LAYOUT_MODEL:', BEST_LAYOUT_MODEL)
print('Exists:', BEST_LAYOUT_MODEL.exists())


## Step 6: Run Final Pipeline With Trained YOLO Cropper

Because this YOLO model has only one class, pass `--target-class 0`.

Replace `TROCR_MODEL` with your fine-tuned TrOCR checkpoint when ready.


In [ ]:
TROCR_MODEL = 'microsoft/trocr-base-handwritten'  # replace with your fine-tuned checkpoint

!python3 pipeline/scripts/run_end_to_end.py \
  --input data/raw/1.jpg \
  --output-dir data/final_demo_yolo \
  --yolo-model "{BEST_LAYOUT_MODEL}" \
  --target-class 0 \
  --ocr-backend trocr \
  --ocr-unit word \
  --trocr-model "{TROCR_MODEL}" \
  --lexicon pipeline/config/drug_lexicon.txt


## Step 7: Inspect Predictions

This table should now reflect crop regions produced by the trained YOLO detector instead of the heuristic fallback.


In [ ]:
import pandas as pd
pred_path = Path('data/final_demo_yolo/predictions.csv')
if pred_path.exists():
    display(pd.read_csv(pred_path).head(30))
else:
    print('No predictions yet:', pred_path)


## Where SAM Fits

SAM is useful for faster annotation or mask refinement, but it does not replace your labeled dataset. A practical path is:

1. Use SAM to suggest masks/boxes for handwritten regions.
2. Manually correct the boxes.
3. Train YOLO on the corrected boxes.
4. Use YOLO for fast repeatable cropping in the final pipeline.

For this project, YOLO bounding boxes are the most direct fix for inaccurate crops.
